# COVID-19 Detection Using Chest X-Ray Images

**Author:** T. Sathvik (1MJ20CS220)  
**Guide:** Mrs. Swasti Sudha, Asst. Professor, Dept. of CSE  
**Institution:** MVJ College of Engineering, Bangalore  
**Academic Year:** 2023-24

---

This notebook trains a custom CNN to classify chest X-ray images into three categories:
- **0 → COVID-19**
- **1 → Normal**
- **2 → Virus (Non-COVID Pneumonia)**

**Dataset:** [COVID CXR Image Dataset Research](https://www.kaggle.com/datasets/sid32laxn/covid-cxr-image-dataset-research)

## 1. Imports

In [ ]:
import os
import pathlib
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import cv2

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, Conv2D, MaxPool2D, Flatten, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

print(f'TensorFlow version: {tf.__version__}')

## 2. Download Dataset from Kaggle

Dataset: **COVID CXR Image Dataset (Research)** — contains `covid`, `normal`, and `virus` chest X-ray folders, sourced from the IEEE COVID-19 dataset.

Upload your `kaggle.json` API key in the cell below, then run the download cell.

In [ ]:
from google.colab import files
files.upload()  # Upload kaggle.json

In [ ]:
os.environ['KAGGLE_CONFIG_DIR'] = '/content'
!kaggle datasets download -d sid321axn/covid-cxr-image-dataset-research
!unzip \*.zip

## 3. Data Exploration

In [ ]:
DATA_DIR = '/content/COVID_IEEE'

for dirpath, dirname, filename in os.walk(DATA_DIR):
    print(f'There are {len(dirname)} directory and {len(filename)} image in {dirpath}')

In [ ]:
data_dir = pathlib.Path(DATA_DIR)
class_names = np.array(sorted([item.name for item in data_dir.glob('*')]))
print('Classes:', class_names)

In [ ]:
def view_image(target_dir, target_class):
    target_folder = target_dir + target_class
    random_image = random.sample(os.listdir(target_folder), 1)
    print(random_image)
    img = mpimg.imread(target_folder + '/' + random_image[0])
    plt.imshow(img, cmap='gray')
    plt.title(target_class)
    plt.axis('off')
    print(f'Image shape {img.shape}')
    return img

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, cls in zip(axes, ['covid', 'normal', 'virus']):
    folder = os.path.join(DATA_DIR, cls)
    img_path = os.path.join(folder, random.choice(os.listdir(folder)))
    img = mpimg.imread(img_path)
    ax.imshow(img, cmap='gray')
    ax.set_title(cls.upper(), fontsize=14)
    ax.axis('off')
plt.suptitle('Sample Chest X-Ray Images per Class', fontsize=16)
plt.tight_layout()
plt.show()

## 4. Data Preprocessing

- Resize all images to 224 × 224
- Assign labels: 0 = COVID, 1 = Normal, 2 = Virus
- Normalise pixel values to [0, 1]
- One-hot encode labels
- 80/20 train-test split

In [ ]:
IMG_SIZE = (224, 224)
data = []
labels = []

# COVID — label 0
for img_name in os.listdir(os.path.join(DATA_DIR, 'covid')):
    img = cv2.imread(os.path.join(DATA_DIR, 'covid', img_name))
    if img is not None:
        data.append(cv2.resize(img, IMG_SIZE))
        labels.append(0)

# Normal — label 1
for img_name in os.listdir(os.path.join(DATA_DIR, 'normal')):
    img = cv2.imread(os.path.join(DATA_DIR, 'normal', img_name))
    if img is not None:
        data.append(cv2.resize(img, IMG_SIZE))
        labels.append(1)

# Virus — label 2
for img_name in os.listdir(os.path.join(DATA_DIR, 'virus')):
    img = cv2.imread(os.path.join(DATA_DIR, 'virus', img_name))
    if img is not None:
        data.append(cv2.resize(img, IMG_SIZE))
        labels.append(2)

print(f'Total images loaded: {len(data)}')
print(f'COVID: {labels.count(0)} | Normal: {labels.count(1)} | Virus: {labels.count(2)}')

In [ ]:
data = np.array(data) / 255.0
image_labels = np.array(labels)

x_train, x_test, y_train, y_test = train_test_split(
    data, image_labels, test_size=0.20, random_state=42
)

NUM_CLASSES = 3
y_train = to_categorical(y_train, num_classes=NUM_CLASSES)
y_test  = to_categorical(y_test,  num_classes=NUM_CLASSES)

print(f'Train: {x_train.shape}, Test: {x_test.shape}')

## 5. Model Definition

Custom 3-block CNN:
- **Block 1:** 2× Conv2D(32) + MaxPool
- **Block 2:** 2× Conv2D(64) + MaxPool
- **Block 3:** 2× Conv2D(128) + 2× MaxPool
- **FC:** Dense(1024) → Dense(256) → Dense(3, softmax)

In [ ]:
model = Sequential()

# Block 1
model.add(Conv2D(input_shape=(224, 224, 3), filters=32, padding='same', kernel_size=(3, 3)))
model.add(Activation('relu'))
model.add(Conv2D(filters=32, padding='same', kernel_size=(3, 3)))
model.add(Activation('relu'))
model.add(MaxPool2D((2, 2)))

# Block 2
model.add(Conv2D(filters=64, padding='same', kernel_size=(3, 3)))
model.add(Activation('relu'))
model.add(Conv2D(filters=64, padding='same', kernel_size=(3, 3)))
model.add(Activation('relu'))
model.add(MaxPool2D((2, 2)))

# Block 3
model.add(Conv2D(filters=128, padding='same', kernel_size=(3, 3)))
model.add(Activation('relu'))
model.add(Conv2D(filters=128, padding='same', kernel_size=(3, 3)))
model.add(Activation('relu'))
model.add(MaxPool2D((2, 2)))
model.add(MaxPool2D((2, 2)))

# Fully Connected
model.add(Flatten())
model.add(Dense(units=1024, activation='relu'))
model.add(Dense(units=256,  activation='relu'))
model.add(Dense(units=3,    activation='softmax'))

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 6. Training

In [ ]:
history = model.fit(
    np.array(x_train).astype('float32'),
    np.array(y_train).astype('float32'),
    validation_split=0.3,
    epochs=15,
    batch_size=32
)

## 7. Training Curves

In [ ]:
loss = pd.DataFrame(history.history)

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(loss['loss'],     label='Training Loss')
plt.plot(loss['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(loss['accuracy'],     label='Training Accuracy')
plt.plot(loss['val_accuracy'], label='Validation Accuracy')
plt.title('Training-Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

## 8. Evaluation

In [ ]:
test_loss, test_acc = model.evaluate(x_test.astype('float32'), y_test.astype('float32'), verbose=0)
print(f'Test Accuracy: {test_acc * 100:.2f}%')
print(f'Test Loss:     {test_loss:.4f}')

In [ ]:
predictions  = model.predict(x_test)
y_pred       = np.argmax(predictions,  axis=1)
y_test_labels = np.argmax(y_test,      axis=1)

CLASS_NAMES = ['covid', 'normal', 'virus']
print(classification_report(y_test_labels, y_pred, target_names=CLASS_NAMES))

In [ ]:
cm = confusion_matrix(y_test_labels, y_pred)
cm_df = pd.DataFrame(cm, columns=CLASS_NAMES, index=CLASS_NAMES)
print('Confusion Matrix:')
print(cm_df)

# Heatmap
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.colorbar(im, ax=ax)
ax.set(xticks=range(NUM_CLASSES), yticks=range(NUM_CLASSES),
       xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
       title='Confusion Matrix',
       ylabel='True Label', xlabel='Predicted Label')
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.tight_layout()
plt.show()

## 9. Save Model

In [ ]:
model.save('model.h5')
print('Model saved as model.h5')

In [ ]:
# Download model to local machine (Google Colab only)
try:
    from google.colab import files
    files.download('model.h5')
except ImportError:
    print('Not running in Colab — model.h5 is saved in the current directory.')

## 10. Single-Image Inference

Predict the class of a new chest X-ray image.

In [ ]:
CLASS_NAMES = ['COVID-19', 'Normal', 'Virus (Non-COVID Pneumonia)']

def predict_image(image_path, model):
    img = cv2.imread(image_path)
    img_resized = cv2.resize(img, (224, 224))
    img_norm = np.array([img_resized]) / 255.0

    pred = model.predict(img_norm)
    pred_class = np.argmax(pred[0])
    confidence = pred[0][pred_class] * 100

    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f'Prediction: {CLASS_NAMES[pred_class]} ({confidence:.1f}%)')
    plt.axis('off')
    plt.show()

    return CLASS_NAMES[pred_class], confidence

# Example usage:
# result, conf = predict_image('/path/to/xray.jpg', model)
# print(f'{result} — {conf:.1f}% confidence')